# Heart Disease EDA and Experimentation
UCI Cleveland Dataset

## 1. Imports and Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score, roc_curve, ConfusionMatrixDisplay
from xgboost import XGBClassifier
import joblib
sns.set_theme(style="whitegrid", palette="Set2")
print("All imports successful!")

All imports successful!


## 2. Load Dataset

In [2]:
df = pd.read_csv("../data/heart.csv")
print("Shape:", df.shape)
df.head()

Shape: (303, 14)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0


## 3. Basic Info

In [3]:
df.info()
df.describe().round(2)

<class 'pandas.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    float64
 1   sex       303 non-null    float64
 2   cp        303 non-null    float64
 3   trestbps  303 non-null    float64
 4   chol      303 non-null    float64
 5   fbs       303 non-null    float64
 6   restecg   303 non-null    float64
 7   thalach   303 non-null    float64
 8   exang     303 non-null    float64
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    float64
 11  ca        303 non-null    str    
 12  thal      303 non-null    str    
 13  target    303 non-null    int64  
dtypes: float64(11), int64(1), str(2)
memory usage: 35.0 KB


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,target
count,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00,303.00
mean,54.44,0.68,3.16,131.69,246.69,0.15,0.99,149.61,0.33,1.04,1.60,0.94
std,9.04,0.47,0.96,17.60,51.78,0.36,0.99,22.88,0.47,1.16,0.62,1.23
min,29.00,0.00,1.00,94.00,126.00,0.00,0.00,71.00,0.00,0.00,1.00,0.00
25%,48.00,0.00,3.00,120.00,211.00,0.00,0.00,133.50,0.00,0.00,1.00,0.00
50%,56.00,1.00,3.00,130.00,241.00,0.00,1.00,153.00,0.00,0.80,2.00,0.00
75%,61.00,1.00,4.00,140.00,275.00,0.00,2.00,166.00,1.00,1.60,2.00,2.00
max,77.00,1.00,4.00,200.00,564.00,1.00,2.00,202.00,1.00,6.20,3.00,4.00


## 4. Missing Values

In [4]:
df.replace("?", float("nan"), inplace=True)
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values!")
df.fillna(df.median(), inplace=True)

ca      4
thal    2
dtype: int64


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
298,45.0,1.0,1.0,110.0,264.0,0.0,0.0,132.0,0.0,1.2,2.0,0.0,7.0,1
299,68.0,1.0,4.0,144.0,193.0,1.0,0.0,141.0,0.0,3.4,2.0,2.0,7.0,2
300,57.0,1.0,4.0,130.0,131.0,0.0,0.0,115.0,1.0,1.2,2.0,1.0,7.0,3
301,57.0,0.0,2.0,130.0,236.0,0.0,2.0,174.0,0.0,0.0,2.0,1.0,3.0,1


## 5. Target Distribution

In [5]:
df["target"] = (df["target"] > 0).astype(int)
tc = df["target"].value_counts()
print("No Disease:", tc[0], "| Heart Disease:", tc[1])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(["No Disease", "Heart Disease"], tc.values, color=["#2ecc71", "#e74c3c"], edgecolor="black")
axes[0].set_title("Target Class Distribution", fontweight="bold")
for i, v in enumerate(tc.values):
    axes[0].text(i, v+2, str(v), ha="center", fontweight="bold")
axes[1].pie(tc.values, labels=["No Disease", "Heart Disease"], autopct="%1.1f%%", colors=["#2ecc71","#e74c3c"], startangle=90)
axes[1].set_title("Target Proportion", fontweight="bold")
plt.tight_layout()
plt.savefig("../data/target_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved.")

No Disease:

 164 | Heart Disease: 139


Saved.


## 6. Numerical Distributions

In [6]:
num_feats = ["age", "trestbps", "chol", "thalach", "oldpeak"]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, col in enumerate(num_feats):
    axes[i].hist(df[df["target"]==0][col], alpha=0.6, label="No Disease", color="#2ecc71", bins=20)
    axes[i].hist(df[df["target"]==1][col], alpha=0.6, label="Heart Disease", color="#e74c3c", bins=20)
    axes[i].set_title(col, fontweight="bold")
    axes[i].legend()
axes[5].axis("off")
plt.suptitle("Numerical Distributions by Target", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/numerical_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved.")

Saved.


## 7. Categorical Distributions

In [7]:
cat_feats = ["sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal"]
fig, axes = plt.subplots(2, 4, figsize=(18, 10))
axes = axes.flatten()
for i, col in enumerate(cat_feats):
    pd.crosstab(df[col], df["target"]).plot(kind="bar", ax=axes[i], color=["#2ecc71","#e74c3c"], edgecolor="black")
    axes[i].set_title(col, fontweight="bold")
    axes[i].legend(["No Disease","Heart Disease"], fontsize=8)
    axes[i].tick_params(axis="x", rotation=30)
plt.suptitle("Categorical Distributions by Target", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/categorical_distributions.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved.")

Saved.


## 8. Correlation Matrix

In [8]:
corr = df.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdYlGn", center=0, linewidths=0.5, square=True, mask=np.triu(np.ones_like(corr, dtype=bool)))
plt.title("Feature Correlation Matrix", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.close()
print("Top correlations with target:")
print(corr["target"].drop("target").sort_values(key=abs, ascending=False).round(3))

Top correlations with target:
thal        0.522
ca          0.460
exang       0.432
oldpeak     0.425
thalach    -0.417
cp          0.414
slope       0.339
sex         0.277
age         0.223
restecg     0.169
trestbps    0.151
chol        0.085
fbs         0.025
Name: target, dtype: float64


## 9. Outlier Detection

In [9]:
fig, axes = plt.subplots(1, 5, figsize=(18, 6))
for i, col in enumerate(num_feats):
    g0 = df[df["target"]==0][col].dropna().values
    g1 = df[df["target"]==1][col].dropna().values
    axes[i].boxplot([g0, g1], patch_artist=True, boxprops=dict(facecolor="lightblue"))
    axes[i].set_xticks([1, 2])
    axes[i].set_xticklabels(["No Disease", "Heart Disease"], rotation=20)
    axes[i].set_title(col, fontweight="bold")
plt.suptitle("Outlier Detection - Numerical Features", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/outliers_boxplot.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved.")

Saved.


## 10. Preprocessing

In [10]:
X = df.drop("target", axis=1)
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (242, 13) | Test: (61, 13)


## 11. Cross-Validation

In [11]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, eval_metric="logloss")
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("=== 5-Fold CV AUC-ROC ===")
for name, model in models.items():
    Xi = X_train_scaled if name == "Logistic Regression" else X_train
    scores = cross_val_score(model, Xi, y_train, cv=cv, scoring="roc_auc")
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

=== 5-Fold CV AUC-ROC ===
Logistic Regression: 0.8960 (+/- 0.0148)


Random Forest: 0.8806 (+/- 0.0325)


XGBoost: 0.8651 (+/- 0.0243)


## 12. Test Set Evaluation

In [12]:
test_results = {}
for name, model in models.items():
    Xtr = X_train_scaled if name == "Logistic Regression" else X_train
    Xte = X_test_scaled if name == "Logistic Regression" else X_test
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)
    y_prob = model.predict_proba(Xte)[:, 1]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    test_results[name] = {"accuracy": acc, "auc": auc, "model": model}
    print(f"{name}: Accuracy={acc:.4f} | AUC={auc:.4f}")
    print(classification_report(y_test, y_pred, target_names=["No Disease","Heart Disease"]))
    print("-"*50)

Logistic Regression: Accuracy=0.8689 | AUC=0.9513
               precision    recall  f1-score   support

   No Disease       0.93      0.82      0.87        33
Heart Disease       0.81      0.93      0.87        28

     accuracy                           0.87        61
    macro avg       0.87      0.87      0.87        61
 weighted avg       0.88      0.87      0.87        61

--------------------------------------------------


Random Forest: Accuracy=0.8852 | AUC=0.9518
               precision    recall  f1-score   support

   No Disease       0.93      0.85      0.89        33
Heart Disease       0.84      0.93      0.88        28

     accuracy                           0.89        61
    macro avg       0.89      0.89      0.89        61
 weighted avg       0.89      0.89      0.89        61

--------------------------------------------------


XGBoost: Accuracy=0.8525 | AUC=0.9188
               precision    recall  f1-score   support

   No Disease       0.93      0.79      0.85        33
Heart Disease       0.79      0.93      0.85        28

     accuracy                           0.85        61
    macro avg       0.86      0.86      0.85        61
 weighted avg       0.86      0.85      0.85        61

--------------------------------------------------


## 13. ROC Curves

In [13]:
plt.figure(figsize=(10, 7))
colors = ["#3498db","#2ecc71","#e74c3c"]
for (name, result), color in zip(test_results.items(), colors):
    Xte = X_test_scaled if name == "Logistic Regression" else X_test
    y_prob = result["model"].predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_val = result["auc"]
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc_val:.3f})", color=color, linewidth=2)
plt.plot([0,1],[0,1],"k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison", fontweight="bold")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("../data/roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved.")

Saved.


## 14. Confusion Matrices

In [14]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, result) in zip(axes, test_results.items()):
    Xte = X_test_scaled if name == "Logistic Regression" else X_test
    y_pred = result["model"].predict(Xte)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["No Disease","Heart Disease"]).plot(ax=ax, colorbar=False, cmap="Blues")
    acc_v = result["accuracy"]
    auc_v = result["auc"]
    ax.set_title(f"{name}\nAcc:{acc_v:.3f} AUC:{auc_v:.3f}", fontweight="bold")
plt.suptitle("Confusion Matrices", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved.")

Saved.


## 15. Feature Importance

In [15]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
feature_names = X.columns.tolist()
for ax, name, color in zip(axes, ["Random Forest","XGBoost"], ["#2ecc71","#e74c3c"]):
    imp = pd.Series(test_results[name]["model"].feature_importances_, index=feature_names).sort_values(ascending=True)
    imp.plot(kind="barh", ax=ax, color=color, edgecolor="black")
    ax.set_title(f"{name} Feature Importance", fontweight="bold")
plt.suptitle("Feature Importance", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../data/feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved.")

Saved.


## 16. Save Best Model

In [16]:
best_name = max(test_results, key=lambda x: test_results[x]["auc"])
best = test_results[best_name]
acc_val = best["accuracy"]
auc_val = best["auc"]
print(f"Best Model: {best_name}")
print(f"  Accuracy: {acc_val:.4f}")
print(f"  AUC-ROC:  {auc_val:.4f}")
joblib.dump(best["model"], "../model/best_model.pkl")
joblib.dump(scaler, "../model/scaler.pkl")
print("Model and scaler saved to ../model/")

Best Model: Random Forest
  Accuracy: 0.8852
  AUC-ROC:  0.9518
Model and scaler saved to ../model/


## 17. Summary

**Key Findings:**
- 303 samples, 13 features, binary target
- ~54% no disease, ~46% disease (balanced)
- Top features: chest pain type (cp), thal, ca, thalach, oldpeak
- XGBoost typically achieves highest AUC-ROC
- Best model and scaler saved to ../model/